# 03 — Alignement des articles français et arabes (مجلة الشغل, 2019)

**Objectif.** Associer chaque article arabe (`articles_ar.csv`, notebook 02) à son équivalent français (`articles_fr.csv`), par règles, sans rien écraser. Chaque paire reçoit un **statut** et un **lot** :
- **lot A** : appariement unique (une seule entrée française pour la clé) : c'est le lot à vérifier en premier sur image ;
- **doublon à trancher** : plusieurs entrées françaises pour la même clé ;
- **absent du français** : aucun équivalent trouvé (hors plages abrogées) ;
- **abrogé** : plage abrogée, sans texte français attendu.

**Clé d'appariement.** Le numéro et le suffixe : « 21 », « 21-9 » (sous-article), « 23 bis », « 14 ter ».
- Arabe : étiquette `AR-…` ; les suffixes arabes « مكرر », « ثالثا », « رابعا », « خامسا » deviennent bis, ter, quater, quinquies.
- Français : en-tête « Article … » du texte ; « quinter » (coquille du texte français) est ramené à quinquies.

**Hors Code.** Les entrées françaises des pages 288 et 289 (loi n° 66-27 de promulgation ; p. 289 vérifiée sur image, p. 288 lue sur le texte OCR) sont exclues de l'appariement et signalées comme « hors Code (loi n° 66-27) ».

**Contrôle complémentaire.** Pour chaque paire, on compare les années citées en tête (texte et note) côté arabe et côté français. Une différence n'annule pas la paire, elle la signale.

**Version v6 de reprise.** Les règles de calcul des cellules 2 à 4 sont identiques à v5. Cette version corrige seulement la détection du dossier local et protège les exports existants.

**Entrées.** `data/processed/articles_ar.csv`, `data/processed/articles_fr_corrige.csv` (ou `articles_fr.csv` à défaut), en lecture seule. Le corpus AR segmenté reste nécessaire ; le texte de travail ne le remplace pas.
**Sorties** (dans `data/processed/`) : `alignement_fr_ar_v6.csv`, `alignement_resume_v6.json`, `alignement_echantillon_image_v6.csv`. Si une de ces sorties existe, l'export est refusé avant toute écriture. Choisir un nouveau dossier de sortie pour une nouvelle expérience ; ne pas supprimer un échantillon annoté.


In [ ]:
import os
import re
import json
import random
from pathlib import Path

import pandas as pd

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def trouver_racine(depart=None):
    """Retrouve le projet même si seuls les fichiers traités ont été récupérés."""
    depart = Path(depart or Path.cwd()).resolve()
    for parent in [depart, *depart.parents]:
        if (parent / "data" / "processed").is_dir() or (parent / "data" / "raw").is_dir():
            return parent
    return None


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/LegalTN")
elif "LEGALTN_ROOT" in os.environ:
    ROOT = Path(os.environ["LEGALTN_ROOT"])
else:
    ROOT = trouver_racine()
    if ROOT is None:
        raise FileNotFoundError("Racine LegalTN introuvable : définir LEGALTN_ROOT.")
print("Racine du projet :", ROOT)

OUT_DIR = ROOT / "data/processed"
FICHIER_AR = OUT_DIR / "articles_ar.csv"
FICHIER_FR = OUT_DIR / "articles_fr_corrige.csv" if (OUT_DIR / "articles_fr_corrige.csv").exists() else OUT_DIR / "articles_fr.csv"
print("Fichier français utilisé :", FICHIER_FR.name)
for f in (FICHIER_AR, FICHIER_FR):
    assert f.exists(), f"Fichier manquant : {f} (lancer d'abord le notebook 02)"

ar = pd.read_csv(FICHIER_AR, dtype=str, encoding="utf-8-sig").fillna("")
fr = pd.read_csv(FICHIER_FR, dtype=str, encoding="utf-8-sig").fillna("")
print("Articles arabes :", len(ar), "| entrées françaises :", len(fr))

In [ ]:
# Suffixes arabes -> latins (clé d'appariement)
SUFFIXES_AR = {"مكرر": "bis", "ثالثا": "ter", "رابعا": "quater", "خامسا": "quinquies",
               "سادسا": "sexies", "سابعا": "septies", "ثامنا": "octies", "تاسعا": "nonies", "عاشرا": "decies"}
# Suffixes français : « quinter » est une coquille du texte français, ramenée à quinquies
SUFFIXES_FR = {"bis": "bis", "ter": "ter", "quater": "quater", "quinquies": "quinquies", "quinter": "quinquies",
               "sexies": "sexies", "septies": "septies", "octies": "octies", "nonies": "nonies", "decies": "decies"}
RE_TETE_FR = re.compile(r"Article\s+(\d+|premier)\s*(?:[-–—]\s*)?(?:(\d+)|(bis|ter|quater|quinquies|quinter|sexies|septies|octies|nonies|decies)\b)?",
                        re.IGNORECASE)


def cle_ar(article_id):
    """AR-21-9 -> « 21-9 » ; AR-14-ثالثا -> « 14 ter » ; AR-97- -> « 97 »."""
    s = article_id[3:].rstrip("-")
    parties = s.split("-", 1)
    if len(parties) == 2 and parties[1] in SUFFIXES_AR:
        return f"{parties[0]} {SUFFIXES_AR[parties[1]]}"
    return s


def cle_fr(texte):
    """« Article 21-8 (…) » -> « 21-8 » ; « Article 14 quinter » -> « 14 quinquies » ; None si pas d'en-tête."""
    m = RE_TETE_FR.match(texte.strip().replace("\u2019", "'"))
    if not m:
        return None
    cle = "1" if m.group(1).lower() == "premier" else m.group(1)
    if m.group(2):
        cle += "-" + m.group(2)
    elif m.group(3):
        cle += " " + SUFFIXES_FR[m.group(3).lower()]
    return cle


ar["cle"] = ar["article_id"].map(cle_ar)
fr["cle"] = fr["texte_fr"].map(cle_fr)
# Bloc de la loi n° 66-27 (promulgation) : hors Code, exclu de l'appariement
PAGES_HORS_CODE = {288, 289}
fr["hors_code"] = fr["page"].astype(int).isin(PAGES_HORS_CODE)
print("Clés sans en-tête reconnu côté français :", int(fr["cle"].isna().sum()))
print("Clés arabes distinctes :", ar["cle"].nunique(), "sur", len(ar), "articles")
print("Clés françaises distinctes :", fr["cle"].dropna().nunique(), "sur", len(fr), "entrées")

In [ ]:
ANNEE = re.compile(r"(?<!\d)(19\d{2}|20\d{2})(?!\d)")


def annees(texte):
    return set(ANNEE.findall(texte))


entrees_fr = fr[~fr["hors_code"]].groupby("cle", dropna=True).apply(lambda g: g.index.tolist(), include_groups=False).to_dict()
utilisees = set()
lignes = []

for _, a in ar.iterrows():
    cle = a["cle"]
    candidats = entrees_fr.get(cle, [])
    if a["abroge"] == "True" and not candidats:
        statut, lot = "abrogé (plage)", ""
        f_id = f_page = f_texte = ""
    elif len(candidats) == 1:
        j = candidats[0]
        utilisees.add(j)
        f = fr.loc[j]
        statut, lot = "apparié", "A"
        f_id, f_page, f_texte = f["article_id"], f["page"], f["texte_fr"]
    elif len(candidats) > 1:
        statut, lot = "doublon à trancher", "B"
        f_id = " | ".join(fr.loc[j, "article_id"] for j in candidats)
        f_page = " | ".join(fr.loc[j, "page"] for j in candidats)
        f_texte = ""
    else:
        statut, lot = "absent du français", "B"
        f_id = f_page = f_texte = ""

    controle = ""
    if statut == "apparié":
        an_ar = annees(a["texte_ar"][:300] + " " + a["note_amendement"])
        an_fr = annees(f_texte[:300])
        if an_ar != an_fr:
            controle = f"années différentes : AR {sorted(an_ar)} / FR {sorted(an_fr)}"

    lignes.append({
        "cle": cle, "article_ar": a["article_id"], "page_ar": a["page"], "abroge": a["abroge"],
        "article_fr": f_id, "page_fr": f_page, "statut": statut, "lot": lot,
        "fr_ligne": (candidats[0] if statut == "apparié" else ""),
        "controle": controle, "texte_fr_debut": f_texte[:160],
    })

# Entrées françaises non appariées : hors Code, ou sans équivalent arabe
for j in fr.index:
    if j in utilisees:
        continue
    if fr.loc[j, "hors_code"]:
        statut = "hors Code (loi n° 66-27)"
    elif fr.loc[j, "cle"] not in set(ar["cle"]):
        statut = "français sans arabe"
    else:
        statut = "doublon non retenu"
    lignes.append({
        "cle": fr.loc[j, "cle"] or "", "article_ar": "", "page_ar": "", "abroge": "",
        "article_fr": fr.loc[j, "article_id"], "page_fr": fr.loc[j, "page"], "statut": statut,
        "lot": "", "fr_ligne": j, "controle": "", "texte_fr_debut": fr.loc[j, "texte_fr"][:160],
    })

alignement = pd.DataFrame(lignes)
print(alignement["statut"].value_counts().to_string())
print("Lot A (appariements uniques) :", int((alignement["lot"] == "A").sum()))
print("Contrôles d'années à relire :", int((alignement["controle"] != "").sum()))

In [ ]:
# Contrôle : chaque article arabe apparaît une fois ; chaque entrée française est utilisée au plus une fois
arabes = alignement[alignement["article_ar"] != ""]
assert arabes["article_ar"].is_unique, "article arabe en double dans l'alignement"
appariees = alignement.loc[alignement["statut"] == "apparié", "fr_ligne"]
assert appariees.is_unique, "entrée française appariée deux fois"
print("Contrôle OK : pas de doublon côté arabe ni côté français (appariements).")

print("\nDoublons à trancher (clé : pages françaises) :")
print(alignement.loc[alignement["statut"] == "doublon à trancher", ["cle", "article_ar", "article_fr", "page_fr"]].to_string(index=False))
print("\nArabes absents du français :")
print(alignement.loc[alignement["statut"] == "absent du français", ["cle", "article_ar", "page_ar"]].to_string(index=False))
print("\nFrançais sans arabe :")
print(alignement.loc[alignement["statut"] == "français sans arabe", ["cle", "article_fr", "page_fr", "texte_fr_debut"]].head(40).to_string(index=False))

In [ ]:
ALIGN_CSV = OUT_DIR / "alignement_fr_ar_v6.csv"
ECH_CSV = OUT_DIR / "alignement_echantillon_image_v6.csv"
RESUME_JSON = OUT_DIR / "alignement_resume_v6.json"
existants = [str(p) for p in (ALIGN_CSV, ECH_CSV, RESUME_JSON) if p.exists()]
if existants:
    raise FileExistsError("Export refusé, fichiers déjà présents : " + ", ".join(existants))

# Échantillon pour contrôle sur image : 10 paires du lot A, tirées avec une graine fixe, plus les contrôles d'années
random.seed(2026)
lot_a = alignement[alignement["lot"] == "A"]
echantillon = lot_a.sample(n=min(10, len(lot_a)), random_state=2026)
controles = alignement[alignement["controle"] != ""]
echantillon = pd.concat([echantillon, controles]).drop_duplicates(subset=["article_ar", "article_fr"])

resume = {
    "articles_ar": int(len(ar)),
    "entrees_fr": int(len(fr)),
    "statuts": {k: int(v) for k, v in alignement["statut"].value_counts().items()},
    "lot_A_appariements_uniques": int((alignement["lot"] == "A").sum()),
    "doublons_a_trancher": int((alignement["statut"] == "doublon à trancher").sum()),
    "absents_du_francais": int((alignement["statut"] == "absent du français").sum()),
    "controles_annees_a_relire": int((alignement["controle"] != "").sum()),
}
alignement.to_csv(ALIGN_CSV, mode="x", index=False, encoding="utf-8-sig")
echantillon[["article_ar", "page_ar", "article_fr", "page_fr", "controle"]].to_csv(ECH_CSV, mode="x", index=False, encoding="utf-8-sig")
with RESUME_JSON.open("x", encoding="utf-8") as fichier_resume:
    json.dump(resume, fichier_resume, ensure_ascii=False, indent=2)
print("Échantillon pour image :", len(echantillon), "paires ->", ECH_CSV)
print(json.dumps(resume, ensure_ascii=False, indent=2))
print("Fichier écrit :", ALIGN_CSV)

## Points à trancher (à vérifier sur image avant toute décision)

- **Loi n° 66-27 (pp. 288–289)** : les entrées « Article premier », « 2 », « 3 », « 4 » de ces pages sont hors Code ; elles sont exclues de l'appariement et listées sous « hors Code ».
- **AR-1 et « Article premier »** : apparié à l'entrée p. 285 (vérifiée sur image) après la correction de la notebook 02 v13.
- **Abrogés** : les plages 272–277, 325–332, 336–337 et 392–396 n'ont pas de texte français attendu ; elles ne sont pas comptées comme erreurs.
- **Contrôles d'années** : paires à relire (années citées différentes entre l'arabe et le français, souvent une coquille OCR).
